# Project Title
## E-Commerce Customer Behavior & Sales Analysis

# Objective
This project analyzes customer behavior and sales patterns in an e-commerce dataset.

Main goals:
- Understand customer purchasing behavior
- Clean and preprocess data for analysis
- Perform exploratory data analysis (EDA) with meaningful visualizations
- Segment customers using K-Means clustering
- Predict Customer Satisfaction using a Random Forest Regressor

# Import Libraries

In [ ]:
# Core libraries
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Visualization and notebook display
from IPython.display import display

# Machine learning and preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Plot styles
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 6)

The required libraries are imported and plotting style is configured for consistent visual output.

# Load Dataset

In [ ]:
# Required placeholder path
file_path = "/content/ecommerce_customer_data.csv"

# Upload dataset in Colab if the file is not already present
if not os.path.exists(file_path):
    from google.colab import files
    import shutil

    print("Please upload your CSV dataset file.")
    uploaded = files.upload()

    if len(uploaded) == 0:
        raise FileNotFoundError("No file uploaded. Please upload a CSV file to continue.")

    uploaded_name = next(iter(uploaded))
    if not uploaded_name.lower().endswith(".csv"):
        raise ValueError("Uploaded file is not a CSV. Please upload a .csv file.")

    if uploaded_name != os.path.basename(file_path):
        shutil.move(uploaded_name, file_path)
        print(f"Uploaded '{uploaded_name}' and saved as '{file_path}'.")
    else:
        print(f"Uploaded file saved at: {file_path}")
else:
    print(f"Dataset already found at: {file_path}")

df = pd.read_csv(file_path)
print(f"Dataset loaded successfully from: {file_path}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
display(df.head())

The notebook now prompts a direct CSV upload in Colab and saves it to /content/ecommerce_customer_data.csv before loading it.

# Data Understanding

In [ ]:
# Basic structure and columns
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(list(df.columns))

# Data types and non-null counts
print("\nData types and non-null counts:")
df.info()

# Summary statistics
print("\nSummary statistics (including categorical columns):")
display(df.describe(include="all").transpose())

# Missing value overview
missing_counts = df.isna().sum().sort_values(ascending=False)
print("\nTop columns with missing values:")
display(missing_counts[missing_counts > 0].head(10))

This step gives an initial profile of the dataset, including data types, summary statistics, and missing values.

# Data Cleaning

In [ ]:
df_clean = df.copy()

# Clean Purchase_Amount by removing currency symbols and spaces, then convert to numeric
if "Purchase_Amount" in df_clean.columns:
    df_clean["Purchase_Amount"] = (
        df_clean["Purchase_Amount"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df_clean["Purchase_Amount"] = pd.to_numeric(df_clean["Purchase_Amount"], errors="coerce")

# Remove duplicate rows
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
removed_duplicates = initial_rows - len(df_clean)

# Handle missing values
# Numeric columns -> median
# Categorical columns -> mode
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
categorical_cols = df_clean.select_dtypes(exclude=[np.number]).columns

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    mode_value = df_clean[col].mode(dropna=True)
    if not mode_value.empty:
        df_clean[col] = df_clean[col].fillna(mode_value.iloc[0])
    else:
        df_clean[col] = df_clean[col].fillna("Unknown")

# Create a numeric income feature for analysis and clustering
income_col_candidates = [c for c in df_clean.columns if "income" in c.lower()]
if income_col_candidates:
    income_col = income_col_candidates[0]
    income_map = {"Low": 1, "Middle": 2, "Medium": 2, "High": 3, "Very High": 4}
    df_clean["Income_Numeric"] = df_clean[income_col].map(income_map)

    # Fallback encoding if mapping does not match categories
    if df_clean["Income_Numeric"].isna().all():
        temp_le = LabelEncoder()
        df_clean["Income_Numeric"] = temp_le.fit_transform(df_clean[income_col].astype(str))
    else:
        df_clean["Income_Numeric"] = df_clean["Income_Numeric"].fillna(df_clean["Income_Numeric"].median())

print(f"Removed duplicate rows: {removed_duplicates}")
print("Total missing values after cleaning:", int(df_clean.isna().sum().sum()))
display(df_clean.head())

Data cleaning completed: missing values handled, duplicate rows removed, and key numeric columns prepared for analysis.

# Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
axes = axes.flatten()

# 1) Histogram of Purchase Amount
if "Purchase_Amount" in df_clean.columns:
    sns.histplot(df_clean["Purchase_Amount"], bins=30, kde=True, ax=axes[0], color="#2a9d8f")
    axes[0].set_title("Histogram of Purchase Amount")
else:
    axes[0].text(0.5, 0.5, "Purchase_Amount not found", ha="center", va="center")
    axes[0].set_title("Histogram of Purchase Amount")

# 2) Income vs Purchase Amount scatter plot
if "Income_Numeric" in df_clean.columns and "Purchase_Amount" in df_clean.columns:
    sns.scatterplot(data=df_clean, x="Income_Numeric", y="Purchase_Amount", alpha=0.6, ax=axes[1], color="#e76f51")
    axes[1].set_title("Income vs Purchase Amount")
    axes[1].set_xlabel("Income (Numeric)")
    axes[1].set_ylabel("Purchase Amount")
else:
    axes[1].text(0.5, 0.5, "Required columns not found", ha="center", va="center")
    axes[1].set_title("Income vs Purchase Amount")

# 3) Correlation heatmap
numeric_df = df_clean.select_dtypes(include=[np.number])
if numeric_df.shape[1] > 1:
    corr = numeric_df.corr()
    sns.heatmap(corr, cmap="YlGnBu", linewidths=0.5, ax=axes[2])
    axes[2].set_title("Correlation Heatmap")
else:
    axes[2].text(0.5, 0.5, "Not enough numeric columns", ha="center", va="center")
    axes[2].set_title("Correlation Heatmap")

# 4) Category-wise analysis
category_col = "Purchase_Category" if "Purchase_Category" in df_clean.columns else None
if category_col and "Purchase_Amount" in df_clean.columns:
    top_cat = (
        df_clean.groupby(category_col)["Purchase_Amount"]
        .mean()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )
    sns.barplot(data=top_cat, x="Purchase_Amount", y=category_col, ax=axes[3], palette="viridis")
    axes[3].set_title("Top Categories by Average Purchase Amount")
else:
    axes[3].text(0.5, 0.5, "Category columns not found", ha="center", va="center")
    axes[3].set_title("Category-wise Analysis")

# 5) Additional insightful plot: Purchase Amount by Channel
channel_col = "Purchase_Channel" if "Purchase_Channel" in df_clean.columns else None
if channel_col and "Purchase_Amount" in df_clean.columns:
    sns.boxplot(data=df_clean, x=channel_col, y="Purchase_Amount", ax=axes[4], palette="Set3")
    axes[4].set_title("Purchase Amount by Channel")
    axes[4].tick_params(axis="x", rotation=25)
else:
    axes[4].text(0.5, 0.5, "Channel column not found", ha="center", va="center")
    axes[4].set_title("Purchase Amount by Channel")

# Extra insight plot (helps interpret the ML target)
if "Customer_Satisfaction" in df_clean.columns:
    sns.countplot(data=df_clean, x="Customer_Satisfaction", ax=axes[5], color="#264653")
    axes[5].set_title("Customer Satisfaction Distribution")
else:
    axes[5].text(0.5, 0.5, "Customer_Satisfaction not found", ha="center", va="center")
    axes[5].set_title("Customer Satisfaction Distribution")

plt.tight_layout()
plt.show()

EDA visualizations highlight spending patterns, income behavior, category performance, and satisfaction trends.

# Customer Segmentation (K-Means)

In [ ]:
# Select relevant numeric features for clustering
cluster_feature_candidates = [
    "Income_Numeric",
    "Purchase_Amount",
    "Frequency_of_Purchase",
    "Product_Rating",
    "Age"
]
cluster_features = [c for c in cluster_feature_candidates if c in df_clean.columns]

if len(cluster_features) < 2:
    raise ValueError("Not enough numeric features available for clustering.")

cluster_df = df_clean[cluster_features].copy().dropna()

# Standardize features before K-Means
scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_df)

# Train K-Means model
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(cluster_scaled)

# Attach cluster labels back to the cleaned data subset
df_clustered = df_clean.loc[cluster_df.index].copy()
df_clustered["Cluster"] = cluster_labels

print("Cluster counts:")
print(df_clustered["Cluster"].value_counts().sort_index())

# Plot clusters using the first two selected features
x_feature = cluster_features[0]
y_feature = cluster_features[1]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clustered, x=x_feature, y=y_feature, hue="Cluster", palette="tab10", s=60, alpha=0.8)
plt.title("Customer Segmentation using K-Means")
plt.legend(title="Cluster")
plt.show()

K-Means clustering groups similar customers, helping identify distinct behavior segments for targeted strategies.

# Machine Learning Model (Prediction)

In [ ]:
df_model = df_clean.copy()

# Identify target column: Customer Satisfaction
target_candidates = [c for c in df_model.columns if "satisfaction" in c.lower()]
if not target_candidates:
    raise ValueError("Customer Satisfaction target column not found in dataset.")
target_col = target_candidates[0]

# Convert target to numeric for regression
df_model[target_col] = pd.to_numeric(df_model[target_col], errors="coerce")
if df_model[target_col].isna().all():
    target_encoder = LabelEncoder()
    df_model[target_col] = target_encoder.fit_transform(df_model[target_col].astype(str))
else:
    df_model[target_col] = df_model[target_col].fillna(df_model[target_col].median())

# Encode categorical predictor columns using LabelEncoder
label_encoders = {}
for col in df_model.columns:
    if col == target_col:
        continue
    if df_model[col].dtype == "object" or str(df_model[col].dtype).startswith("category"):
        le = LabelEncoder()
        df_model[col] = le.fit_transform(df_model[col].astype(str))
        label_encoders[col] = le

# Convert boolean columns to numeric
bool_cols = df_model.select_dtypes(include=["bool"]).columns
for col in bool_cols:
    df_model[col] = df_model[col].astype(int)

# Fill any remaining missing values
for col in df_model.columns:
    if df_model[col].dtype.kind in "biufc":
        df_model[col] = df_model[col].fillna(df_model[col].median())
    else:
        df_model[col] = df_model[col].fillna(df_model[col].mode().iloc[0])

# Prepare features and target
drop_cols = [target_col]
if "Customer_ID" in df_model.columns:
    drop_cols.append("Customer_ID")  # Identifier column is not useful for prediction

X = df_model.drop(columns=drop_cols, errors="ignore")
y = df_model[target_col]

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=250, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)

print("Model training completed successfully.")
print("Training samples:", X_train.shape[0], "| Testing samples:", X_test.shape[0])
print("Feature count:", X.shape[1])

A Random Forest Regressor is trained to predict Customer Satisfaction using an 80/20 split.

# Results & Evaluation

In [ ]:
# Evaluate model performance
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("=== Model Evaluation ===")
print(f"R2 Score: {r2:.4f}")
print(f"Mean Squared Error: {mse:.4f}")

# Plot actual vs predicted values
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7, color="#1d3557")
plt.xlabel("Actual Customer Satisfaction")
plt.ylabel("Predicted Customer Satisfaction")
plt.title("Actual vs Predicted Customer Satisfaction")
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--")
plt.show()

# Show top feature importances
feature_importance = (
    pd.Series(rf_model.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 5))
sns.barplot(x=feature_importance.values, y=feature_importance.index, color="#457b9d")
plt.title("Top 10 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

The model performance is summarized using R2 Score and Mean Squared Error, followed by visual diagnostics.

# Conclusion
This notebook provides a complete workflow for e-commerce customer analysis:
- Cleaned and prepared customer transaction data
- Performed EDA with meaningful visual insights
- Segmented customers into behavior-based clusters using K-Means
- Built and evaluated a Random Forest model to predict Customer Satisfaction

This analysis can support business decisions in personalization, customer retention, and campaign targeting.